# 21 — Qwen2.5-7B-Instruct few-shot (esempi recuperati dal train)

Stesso modello del notebook 07 (`qwen2.5:7b-instruct` via ollama, `max_tokens=200`,
`temperature=0.3`), ma invece della richiesta zero-shot il prompt contiene **k esempi**
(articolo → riassunto umano) presi dal **train**: per ogni cluster da riassumere si recuperano i
k cluster del train più simili (coseno fra embedding `all-mpnet-base-v2`). Slug `qwen_fewshot`.
Ambiti: `SCOPE='sample'` (smoke test), `SCOPE='test'` (5.610 righe) e `SCOPE='test_budgetref'`.

Origine: i notebook di Federica `notebooks/llm/20_preliminary_fs_test.ipynb` (esplorazione di
embedding, k e prompt) e `notebooks/llm/21_fsprompt_qwen.ipynb` (la sua corsa completa),
archiviati con i loro output. Il prompt è il suo, copiato alla lettera in `su.prompt_fewshot`;
k e variante sono fissati dal pilota del notebook 20 (su righe di **validation**, così la
scelta non guarda il test). La sua corsa non è stata importata: usava row_id locali (0..5611,
con le due righe d'intestazione Orange lette come dati) invece degli indici globali di
`complete.tab`, e non impostava il contesto di ollama.

Differenze deliberate rispetto al notebook 07, tutte dichiarate nella `config` del JSON:
- **API nativa di ollama** (`/api/chat` via `su.genera_ollama_nativo`) invece dell'endpoint
  OpenAI-compatibile: è l'unico canale che accetta `num_ctx` per richiesta. `num_ctx=32768`
  (il contesto nativo di Qwen2.5); una richiesta che satura il contesto **fallisce** invece di
  essere troncata in silenzio da ollama;
- un solo messaggio `user` (niente `system`, niente `/no_think`), come nei notebook di Federica;
- troncamenti che tengono il prompt nel contesto: articolo di ogni esempio a 1.000 parole
  (il riassunto dell'esempio resta intero), documento da riassumere a 16.000 parole
  (7 cluster del test su 5.610).

ℹ️ **Ambito `test_budgetref`** (issue #16, protocollo *length-matched*): stesse righe e stessi
vicini dell'ambito `test`, ma nel prompt la lunghezza obiettivo del cluster — la lunghezza del
riassunto di riferimento (`su.budget_riferimento`). Il prompt a budget è **derivato** da `new`
(`su.prompt_fewshot_budget`): la sola riga che chiedeva di imitare stile e lunghezza degli
esempi diventa *«of about N words, matching the style of the k example summaries. The summary
must be at least N words long»*, con N = `FATTORE_RICHIESTA` × budget; `max_tokens` sale a
`MAX_TOKENS_BUDGET = 1500`. Serve perché nell'ambito `test` il few-shot resta lontano dalla
banda: sul pilota del notebook 20 k4_new scrive in mediana 0,54× il riferimento (17% delle righe
in [0,8; 1,25]) — gli esempi non trasmettono la lunghezza, e `max_tokens=200` la limita a ~170
parole. Il fattore è calibrato con un pilota di 30 righe (`SUMM_LIMIT=30`), come per gli LLM
zero-shot: 1,2 (il valore di qwen zero-shot) lascia il 23% delle righe sotto banda, 1,4 porta
la mediana a 1,09× e l'87% delle righe in banda; il tetto a 1,25× si applica comunque a valle (`scripts/applica_budget.py`). È
un'informazione gold — la sola lunghezza — e i numeri vanno letti come «selezione dei contenuti
a parità di lunghezza», non come prestazione in produzione.

In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip install sentence-transformers

C:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Configurazione ---------------------------------------------------------
import os

import summ_utils as su

METODO     = 'qwen_fewshot'
# 'sample' (default), 'test' o 'test_budgetref'; run_benchmark_test.py imposta SUMM_SCOPE
SCOPE      = os.environ.get('SUMM_SCOPE', 'sample')
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
# es. 3 per uno smoke test rapido; None = tutti; SUMM_LIMIT usato da run_benchmark_test.py
LIMIT      = int(os.environ['SUMM_LIMIT']) if 'SUMM_LIMIT' in os.environ else None
MODELLO    = 'qwen2.5:7b-instruct'  # tag ollama, lo stesso del notebook 07

MAX_TOKENS  = 200         # come il notebook 07
TEMPERATURE = 0.3
# Fissati dal pilota del notebook 20 (validation, 100 righe): unica configurazione
# sopra k=0 su ROUGE-1, ROUGE-2 e BERTScore insieme, e la meno contaminata dagli esempi
K           = 4
VARIANTE    = 'new'
NUM_CTX     = su.NUM_CTX_FEWSHOT
PAROLE_ESEMPIO   = su.PAROLE_ESEMPIO_FEWSHOT
MAX_PAROLE_DOC   = su.MAX_PAROLE_DOCUMENTO_FEWSHOT
ETICHETTA   = 'Qwen few-shot '

# Ambito a lunghezza del riferimento (issue #16): con SUMM_SCOPE='test_budgetref' il budget
# in parole del cluster entra nel prompt (su.prompt_fewshot_budget, derivato da 'new') e il
# limite in token sale a MAX_TOKENS_BUDGET; None = ambito test, comportamento invariato.
BUDGET = su.budget_riferimento if su.budget_attivo(SCOPE) else None
# Calibrato su un pilota di 30 righe del test (2026-09-27): con 1,2 (il valore di qwen
# zero-shot) mediana 0,91 x budget e 23% delle righe sotto banda — gli esempi rendono il
# modello piu' conservativo; con 1,4 mediana 1,09 x, 13% sotto banda, 87% in banda dopo il
# tetto. E' una calibrazione, non un principio.
FATTORE_RICHIESTA = 1.4
MAX_TOKENS_BUDGET = 1500

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH  = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH     = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
FEWSHOT_DIR  = BASE / 'results' / 'fewshot'
# I vicini dipendono solo dalle righe: l'ambito a budget riusa quelli del test
VICINI_PATH  = FEWSHOT_DIR / f'{METODO}_vicini_{su.split_base(SCOPE)}.tsv'   # committato
CACHE_EMB    = BASE / 'results' / 'embeddings_cache'          # non versionato (~140 MB)

def esempi_scope():
    """Iterabile degli esempi dell'ambito selezionato (riusato anche dalla valutazione)."""
    if SCOPE == 'sample':
        return su.carica_campione(SAMPLE_PATH)
    if su.split_base(SCOPE) == 'test':   # 'test' e 'test_budgetref' leggono le stesse righe
        return su.itera_split(P['complete_tab'], 'test')
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

print(f'Modello : {MODELLO} (API nativa ollama, num_ctx={NUM_CTX})')
print(f'Ambito  : {SCOPE} | k={K} | prompt={VARIANTE}')
print(f'Budget  : {"riferimento x " + str(FATTORE_RICHIESTA) + " nel prompt, max_tokens=" + str(MAX_TOKENS_BUDGET) if BUDGET else "nessuno, max_tokens=" + str(MAX_TOKENS)}')
print(f'Output  : {OUT_PATH}')

Modello : qwen2.5:7b-instruct (API nativa ollama, num_ctx=32768)
Ambito  : test | k=4 | prompt=new
Output  : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\qwen_fewshot_test.tsv


## Retrieval degli esempi

Gli embedding del train (44.880 cluster) si calcolano una volta e restano in cache in
`results/embeddings_cache/` (non versionata). I vicini finiscono in
`results/fewshot/qwen_fewshot_vicini_{scope}.tsv` (row_id → row_id del train, rango,
similarità), che **si committa**: a parità di file la generazione non dipende dagli embedding.
Se il TSV esiste già viene riusato; per rifare il retrieval va cancellato.

⚠️ Nell'ambito `sample` le righe del campione che vengono dal train sono **escluse**: il pool
degli esempi è l'intero train, e quelle righe ritroverebbero se stesse come esempio (è il
riassunto di riferimento dato al modello). Lo smoke test gira quindi sulle sole righe val/test
del campione.

In [3]:
righe = [es for es in esempi_scope() if es['split'] != 'train']
print(f'{len(righe)} righe da riassumere')

if VICINI_PATH.exists():
    VICINI = su.carica_vicini(VICINI_PATH)
    print(f'Vicini dal file: {VICINI_PATH}')
else:
    VICINI = su.calcola_vicini(P['complete_tab'], righe, K, CACHE_EMB, f'{METODO}_{su.split_base(SCOPE)}', VICINI_PATH)
assert all(len(VICINI[r['row_id']]) >= K for r in righe), 'TSV dei vicini con meno di K vicini'

ESEMPI_TRAIN = su.carica_esempi_train(P['complete_tab'],
                                      {v for r in righe for v, _ in VICINI[r['row_id']][:K]},
                                      PAROLE_ESEMPIO)
print(f'{len(ESEMPI_TRAIN)} esempi del train caricati')
sim1 = sorted(VICINI[r['row_id']][0][1] for r in righe)
print(f'Similarita\' del 1o vicino: mediana {sim1[len(sim1) // 2]:.3f}, max {sim1[-1]:.3f}')

5610 righe da riassumere


[train] embedding dalla cache (train_all-mpnet-base-v2_2e3005dfe4559857.npy)


[qwen_fewshot_test] calcolo embedding di 5610 testi...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5441.61it/s]

Vicini: C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\fewshot\qwen_fewshot_vicini_test.tsv (5610 righe x 4)


15492 esempi del train caricati
Similarita' del 1o vicino: mediana 0.734, max 1.000


## Generazione dei riassunti

Il modello gira **fuori dal notebook**, nel server locale di ollama (`ollama pull
qwen2.5:7b-instruct`). Il prompt di ogni riga si costruisce qui, per `row_id`, e il ciclo
condiviso riceve direttamente il prompt (`prepara` è l'identità): la mappa riga → esempi non
dipende dal testo del documento. Una risposta vuota o un contesto saturo sollevano
un'eccezione: la riga è registrata come errore e **non** scritta nel TSV (ritentabile).

⚠️ Ripresa: come negli altri notebook, rieseguire su un TSV esistente aggiunge solo le righe
mancanti. Se cambiano `K`, `VARIANTE` o i troncamenti, **cancellare prima** il TSV.

In [4]:
import statistics

def prompt_riga(es):
    esempi = [ESEMPI_TRAIN[v] for v, _ in VICINI[es['row_id']][:K]]
    documento = su.tronca_parole(su.prepara_documento(es['document']), MAX_PAROLE_DOC)
    if BUDGET is None:
        return su.prompt_fewshot(esempi, documento, VARIANTE)
    richiesto = int(round(FATTORE_RICHIESTA * BUDGET(es)))
    return su.prompt_fewshot_budget(esempi, documento, richiesto, VARIANTE)

def righe_con_prompt():
    for es in righe:
        yield {**es, 'document': prompt_riga(es)}

TOKEN_PROMPT = []   # diagnostica di questa esecuzione (non sopravvive a una ripresa)

def genera(prompt):
    testo, n_prompt = su.genera_ollama_nativo([{'role': 'user', 'content': prompt}], MODELLO,
                                              MAX_TOKENS_BUDGET if BUDGET else MAX_TOKENS,
                                              TEMPERATURE, num_ctx=NUM_CTX)
    TOKEN_PROMPT.append(n_prompt)
    return testo

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(righe_con_prompt(), scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA, prepara=lambda p: p)
scrittore.chiudi()
if TOKEN_PROMPT:
    print(f'Token del prompt: mediana {statistics.median(TOKEN_PROMPT):.0f}, '
          f'max {max(TOKEN_PROMPT)} (contesto {NUM_CTX})')

Qwen few-shot [1] media 17.0 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [2] media 11.5 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [3] media 10.8 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [10] media 9.3 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [20] media 8.7 s/esempio (saltati 0 gia' fatti)


...

Qwen few-shot [5570] media 9.3 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [5580] media 9.3 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [5590] media 9.3 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [5600] media 9.3 s/esempio (saltati 0 gia' fatti)


Qwen few-shot [5610] media 9.3 s/esempio (saltati 0 gia' fatti)
Qwen few-shot Completato: 5610 nuovi, 0 saltati, 0 errori, 52207 s totali
Token del prompt: mediana 7850, max 31330 (contesto 32768)


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare. Stesse metriche e stessa
normalizzazione di tutti i metodi del benchmark. Il BERTScore **non** si calcola qui: lo
aggiunge il notebook 13 (incrementale), il G-Eval il notebook 14 / `scripts/run_geval.py`.

In [5]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = esempi_scope()
config = {'modello': MODELLO,
          'backend': 'ollama (API nativa /api/chat)',
          'max_tokens': MAX_TOKENS_BUDGET if BUDGET else MAX_TOKENS,
          'temperature': TEMPERATURE, 'num_ctx': NUM_CTX,
          'budget_parole': f'lunghezza del riferimento nel prompt x {FATTORE_RICHIESTA} (issue #16)' if BUDGET else None,
          'k': K, 'variante_prompt': VARIANTE,
          'embedding': su.MODELLO_EMBEDDING_FEWSHOT, 'pool_esempi': 'split train di complete.tab',
          'parole_esempio': PAROLE_ESEMPIO, 'max_parole_documento': MAX_PAROLE_DOC,
          'vicini': VICINI_PATH.name,
          'note': 'few-shot con esempi recuperati (nearest neighbour coseno), prompt di '
                  'Federica (notebooks/llm/20); k e variante dal pilota del notebook 20'}

righe_metriche, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                              P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

Metriche per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\qwen_fewshot_test_per_example.csv (5610 righe)
Metriche aggregate   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\qwen_fewshot_test_aggregate.json
{
  "rouge1_f1": 0.33743391178277155,
  "rouge1_p": 0.4499197586151571,
  "rouge1_r": 0.2785396905049044,
  "rouge2_f1": 0.1075359614499925,
  "rouge2_p": 0.1517936874069521,
  "rouge2_r": 0.086734166341443,
  "rougeL_f1": 0.18773373193967557,
  "rougeL_p": 0.26843256616613853,
  "rougeL_r": 0.15085113722657517,
  "bleu": 0.05509305578467182,
  "meteor": 0.29755306079972266,
  "parole_generate": 115.88663101604278
}

Medie per split:
  test  (n=5610): ROUGE-1 F1 = 0.337


## Ispezione qualitativa

In [6]:
su.mostra_esempi(esempi_scope(), riassunti, quanti=2)

--- row_id=50491 (split=test) ---
GENERATO   : Voters in 11 states will choose their governors tonight, with Republicans poised to gain at least one seat and possibly extend their control to over two-thirds of the nation's top state offices. Eight current Democratic governorships are considered vulnerable, while three Republican-held seats are competitive. Democrats are expected to hold on to their seats in West Virginia and Missouri and win in Vermont and Delaware. Key races include North Carolina, where Republican Pat McCrory is favored to
RIFERIMENTO: It's a race for the governor's mansion in 11 states today, and the GOP could end the night at the helm of more than two-thirds of the 50 states. The GOP currently controls 29 of the country's top state offices; it's expected to keep the three Republican ones that are up for grabs (Utah, North Dakota, and Indiana), and wrest North Carolina from the Dems. That brings its toll to 30, with the potential to take three more, reports NPR. Rac